## Student setup

Run the modules in order: M1 creates the handoff dataset consumed by M2, M3, and M4. Keep the master dataset in approved private storage; it is intentionally excluded from this repository. Set `MEDML_MASTER_DATASET_PATH` for Module 1 and `MEDML_OUTPUT_DIR` if outputs should persist outside the repository.


# M1 | MIMIC-IV-ED dataset exploration

This notebook is the evidence-first introduction to the extracted MIMIC-IV-ED teaching table. We establish what one row represents, inspect the 120 variables, calculate basic statistics, identify missingness and data-quality issues, verify the three analysis outcomes, and document which fields are not eligible predictors.

**Dataset:** `private authorized master dataset (master_dataset_new.csv)`

## Learning objectives and route through the notebook

By the end, students should be able to:

1. State the row grain, identifiers, observation window, and provenance of the table.
2. Classify variables as identifiers, timing fields, demographics, triage observations, prior history, outcomes, or downstream measurements.
3. Report mean, standard deviation, median, IQR, percentiles, frequencies, and missingness without hiding skew.
4. Separate a descriptive association from a prediction feature and identify leakage.
5. Reproduce the definitions of ED LOS, critical outcome, and hospitalization.

Every quantitative claim in later modules should be traceable to a computation like the ones below.

## Provenance and interpretation boundaries

The CSV is an extracted, de-identified teaching dataset derived from MIMIC-IV-ED. It is suitable for learning data analysis and reproducible modeling workflows; it is not a clinical decision-support dataset. The source repository and benchmark notebooks supplied with the workspace are methodological references. We recompute all summaries from the local CSV rather than copying historical outputs.

A useful habit is to ask four questions before looking at a model: **Who is represented? What is one row? When was each variable available? What exactly is the label?**

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
import os

repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (
        candidate
        for candidate in repo_candidates
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir()
    ),
    Path.cwd(),
)
output_override = os.getenv("MEDML_OUTPUT_DIR")
OUTPUT_DIR = Path(output_override).expanduser() if output_override else REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

master_override = os.getenv("MEDML_MASTER_DATASET_PATH")
candidate_paths = []
if master_override:
    candidate_paths.append(Path(master_override).expanduser())
candidate_paths.extend(
    [
        Path("/content/drive/MyDrive/MedML_Toolbox/master_dataset_new.csv"),
        REPO_ROOT / "data" / "private" / "master_dataset_new.csv",
        REPO_ROOT / "data" / "master_dataset_new.csv",
    ]
)
DATA_PATH = next((candidate.resolve() for candidate in candidate_paths if candidate.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Private M1 data not found. Set MEDML_MASTER_DATASET_PATH to master_dataset_new.csv "
        "or place it in the approved private Google Drive path."
    )
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")



In [ ]:
pd.set_option("display.max_columns", 200)
display(df.head(5))

## 1. Row grain, identifiers, and repeat visits

The row grain is one emergency-department stay. `stay_id` should identify that stay, while `subject_id` identifies the person and can appear more than once. `hadm_id` links an ED stay to a hospital admission when it is present. These fields support auditing and grouping, but identifiers are not automatically valid predictors.

In [ ]:
identifier_summary = pd.DataFrame({
    "quantity": [
        "rows", "columns", "unique stay_id", "unique subject_id",
        "subjects with multiple stays", "non-missing hadm_id values",
    ],
    "value": [
        len(df), df.shape[1], df["stay_id"].nunique(), df["subject_id"].nunique(),
        int((df["subject_id"].value_counts() > 1).sum()), df["hadm_id"].nunique(dropna=True),
    ],
})
display(identifier_summary)
assert df["stay_id"].is_unique
assert df["stay_id"].notna().all()

## 2. Variable dictionary: read the table as a measurement system

The extracted columns form several families. The family is part of the meaning: `triage_*` fields are clinical observations at triage, prior-utilization counts summarize history before the index stay, `ed_*_last` fields describe later ED measurements, and `outcome_*` fields are labels or downstream events.

The dictionary below is generated from the actual columns, so it remains useful if a future extraction adds or removes a field.

In [ ]:
variable_groups = {
    "identifiers_and_timing": [
        "index", "subject_id", "hadm_id", "stay_id", "intime", "outtime", "admittime",
        "dischtime", "edregtime", "edouttime", "in_year", "anchor_age", "anchor_year",
    ],
    "demographics_and_arrival": [
        "age", "gender", "race", "ethnicity", "insurance", "arrival_transport", "disposition",
    ],
    "outcomes_and_downstream_events": [
        "ed_los", "ed_los_hours", "outcome_inhospital_mortality", "outcome_icu_transfer_12h",
        "time_to_icu_transfer_hours", "outcome_hospitalization", "outcome_critical",
        "outcome_ed_revisit_3d",
    ],
    "triage_measurements": [
        "triage_temperature", "triage_heartrate", "triage_resprate", "triage_o2sat",
        "triage_sbp", "triage_dbp", "triage_pain", "triage_acuity",
    ],
    "prior_utilization": [
        "n_ed_30d", "n_ed_90d", "n_ed_365d", "n_hosp_30d", "n_hosp_90d", "n_hosp_365d",
        "n_icu_30d", "n_icu_90d", "n_icu_365d",
    ],
    "complaint_flags": [column for column in df.columns if column.startswith("chiefcom_")],
    "charlson_flags": [column for column in df.columns if column.startswith("cci_")],
    "elixhauser_flags": [column for column in df.columns if column.startswith("eci_")],
    "later_ed_measurements_and_medications": [
        column for column in df.columns if column.startswith("ed_") and column not in {"ed_los", "ed_los_hours"}
    ] + ["n_med", "n_medrecon"],
}
dictionary_rows = []
for family, columns in variable_groups.items():
    for column in columns:
        if column in df.columns:
            dictionary_rows.append({
                "family": family,
                "variable": column,
                "dtype": str(df[column].dtype),
                "non_missing": int(df[column].notna().sum()),
                "missing_pct": df[column].isna().mean() * 100,
                "unique_non_missing": df[column].nunique(dropna=True),
            })
variable_dictionary = pd.DataFrame(dictionary_rows)
display(variable_dictionary.head(50).round(2))

In [ ]:
# Preserve Fahrenheit source values and add Celsius equivalents.
for column in ["triage_temperature", "ed_temperature_last"]:
    if column in df.columns:
        df[f"{column}_celsius"] = (df[column] - 32) * 5 / 9

display(df[["triage_temperature", "triage_temperature_celsius",
            "ed_temperature_last", "ed_temperature_last_celsius"]].head())

In [ ]:
# Display summary of the variable dictionary
print(f"Total variables documented: {len(variable_dictionary)}")
print(f"\nVariables by family:")
family_counts = variable_dictionary['family'].value_counts()
display(family_counts)

print(f"\nVariables with highest missingness:")
high_missing = variable_dictionary.nlargest(10, 'missing_pct')[['variable', 'family', 'missing_pct']]
display(high_missing.round(2))

### Exercise: timing and availability

For each of these fields, write **available at triage**, **available during the first hour**, **available only after the stay**, or **uncertain**: `triage_acuity`, `n_ed_365d`, `ed_heartrate_last`, `ed_los_hours`, `disposition`, `outcome_hospitalization`. Explain why a field's clinical plausibility is not enough to make it a valid predictor.

In [ ]:
numeric_variables = [
    "age", "triage_acuity", "triage_temperature_celsius", "triage_heartrate", "triage_resprate",
    "triage_o2sat", "triage_sbp", "triage_dbp", "triage_pain", "ed_los_hours",
    "n_ed_365d", "n_hosp_365d", "n_icu_365d", "ed_temperature_last_celsius","ed_heartrate_last"
]
numeric_variables = [column for column in numeric_variables if column in df.columns]
numeric_summary = df[numeric_variables].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.90, 0.99]).T
numeric_summary["iqr"] = numeric_summary["75%"] - numeric_summary["25%"]
numeric_summary["missing_pct"] = df[numeric_variables].isna().mean() * 100
display(numeric_summary[["count", "mean", "std", "min", "1%", "25%", "50%", "75%", "90%", "99%", "max", "iqr", "missing_pct"]].round(2))

## 3. Categorical frequencies and representation

Frequencies describe who and what is represented in this extract. They do not establish that a category is clinically meaningful, fair, or transportable. Small categories and missing categories deserve explicit review before encoding.

In [ ]:
categorical_variables = [
    "gender", "race", "ethnicity", "insurance", "arrival_transport", "disposition", "triage_acuity",
]
categorical_tables = {}
for variable in categorical_variables:
    if variable not in df.columns:
        continue
    table = (
        df[variable].value_counts(dropna=False)
        .rename_axis(variable)
        .reset_index(name="stays")
    )
    table["percent"] = table["stays"] / len(df) * 100
    categorical_tables[variable] = table
    print(f"\n{variable}")
    display(table.head(15).round(2))

## 4. Missingness is part of the data-generating process

A blank can mean a test was not ordered, a value was not documented, a measurement fell outside a time window, or the extraction failed. It is not automatically normal physiology. We inspect missingness before choosing imputation, and later modeling pipelines add missingness indicators for numeric features.

In [ ]:
missingness = (
    df.isna().mean().mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)
display(missingness.head(25).round(2))
plt.figure(figsize=(10, 6))
sns.barplot(data=missingness.head(20).reset_index(names="variable"), x="missing_pct", y="variable", color="#0d9488")
plt.xlabel("Missing values (%)")
plt.ylabel("")
plt.title("Twenty variables with the highest missingness")
plt.tight_layout()
plt.show()

## 5. Vital-sign missing-value imputation exercise: compare before applying

4 imputers tested: median imputation, K-nearest neighbors (KNN), iterative chained equations, and an autoencoder..

The validation design is artificial masking: start with rows where all candidate vitals are observed, hide 20% of the observed cells, impute them, and score only the cells that were deliberately hidden. This gives us a known reference value. It does **not** prove that every real missing value is recoverable.

The source workflow also defined broad physiologic ranges. Values outside an extreme range are treated as missing for this imputation exercise; values between the extreme and clinically usable boundary are clipped to the nearest boundary. These rules create a cleaned working copy only.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

source_vitals = [
    "triage_temperature_celsius", "triage_heartrate", "triage_resprate",
    "triage_o2sat", "triage_sbp", "triage_dbp", "triage_pain", "triage_acuity",
    "ed_temperature_last_celsius", "ed_heartrate_last", "ed_resprate_last",
    "ed_o2sat_last", "ed_sbp_last", "ed_dbp_last", "ed_pain_last",
]
source_vitals = [column for column in source_vitals if column in df.columns]

vital_ranges = {
    "temperature": (14.2, 26, 45, 47),
    "heartrate": (0, 0, 350, 390),
    "resprate": (0, 0, 300, 330),
    "o2sat": (0, 0, 100, 150),
    "sbp": (0, 0, 375, 375),
    "dbp": (0, 0, 375, 375),
    "pain": (0, 0, 10, 10),
    "acuity": (1, 1, 5, 5),
}

def clean_vitals_for_imputation(vitals):
    cleaned = vitals.copy()
    for column in cleaned.columns:
        vital_type = column.split("_", 2)[1]
        outlier_low, valid_low, valid_high, outlier_high = vital_ranges[vital_type]
        if vital_type == "temperature" and cleaned[column].median(skipna=True) > 60:
            outlier_low, valid_low, valid_high, outlier_high = tuple(
                value * 9 / 5 + 32 for value in (outlier_low, valid_low, valid_high, outlier_high)
            )
        cleaned.loc[cleaned[column].lt(outlier_low) | cleaned[column].gt(outlier_high), column] = np.nan
        cleaned[column] = cleaned[column].clip(lower=valid_low, upper=valid_high)
    return cleaned

vitals_cleaned = clean_vitals_for_imputation(df[source_vitals])
imputation_min_values = []
imputation_max_values = []
for column in source_vitals:
    vital_type = column.split("_", 2)[1]
    _, valid_low, valid_high, _ = vital_ranges[vital_type]
    if vital_type == "temperature" and vitals_cleaned[column].median(skipna=True) > 60:
        valid_low, valid_high = (value * 9 / 5 + 32 for value in (valid_low, valid_high))
    imputation_min_values.append(valid_low)
    imputation_max_values.append(valid_high)

before_imputation = pd.DataFrame({
    "variable": source_vitals,
    "raw_missing": [int(df[column].isna().sum()) for column in source_vitals],
    "range_cleaned_missing": [int(vitals_cleaned[column].isna().sum()) for column in source_vitals],
    "raw_missing_pct": [df[column].isna().mean() * 100 for column in source_vitals],
})
display(before_imputation.round(2))
print(f"Complete rows available for validation: {int(vitals_cleaned.dropna().shape[0]):,}")

In [ ]:
# Hide known values, fit each method on the masked table, and score hidden cells only.
complete_vitals = vitals_cleaned.dropna()
if complete_vitals.empty:
    complete_vitals = df[source_vitals].dropna()
if complete_vitals.empty:
    raise ValueError("No complete vital-sign rows are available for masked-cell validation. Re-run the preparation cell or inspect source_vitals.")
validation_size = min(10_000, len(complete_vitals))
validation_truth = complete_vitals.sample(validation_size, random_state=42)
mask_rng = np.random.default_rng(42)
artificial_mask = mask_rng.random(validation_truth.shape) < 0.20
masked_vitals = validation_truth.mask(artificial_mask)

imputers = {
    "median": SimpleImputer(strategy="median"),
    "knn": KNNImputer(n_neighbors=5),
    "iterative": IterativeImputer(
        random_state=0,
        max_iter=10,
        min_value=imputation_min_values,
        max_value=imputation_max_values,
    ),
}
comparison_rows = []
for method, imputer in imputers.items():
    imputed = pd.DataFrame(imputer.fit_transform(masked_vitals), columns=source_vitals, index=masked_vitals.index)
    true_values = validation_truth.to_numpy()[artificial_mask]
    predicted_values = imputed.to_numpy()[artificial_mask]
    comparison_rows.append({
        "method": method,
        "mse": mean_squared_error(true_values, predicted_values),
        "mae": mean_absolute_error(true_values, predicted_values),
        "masked_cells": int(artificial_mask.sum()),
    })

# The source notebook used a Keras autoencoder; this dependency-free MLP version keeps the same reconstruction idea.
autoencoder_scaler = StandardScaler()
masked_filled = masked_vitals.fillna(masked_vitals.median())
masked_scaled = autoencoder_scaler.fit_transform(masked_filled)
autoencoder = MLPRegressor(hidden_layer_sizes=(10,), max_iter=100, random_state=0, early_stopping=True)
autoencoder.fit(masked_scaled, masked_scaled)
autoencoded = pd.DataFrame(
    autoencoder_scaler.inverse_transform(autoencoder.predict(masked_scaled)),
    columns=source_vitals,
    index=masked_vitals.index,
)
autoencoded_values = autoencoded.to_numpy()[artificial_mask]
comparison_rows.append({
    "method": "autoencoder",
    "mse": mean_squared_error(true_values, autoencoded_values),
    "mae": mean_absolute_error(true_values, autoencoded_values),
    "masked_cells": int(artificial_mask.sum()),
})

imputation_comparison = pd.DataFrame(comparison_rows).sort_values(["mse", "mae"]).reset_index(drop=True)
display(imputation_comparison.round(4))
best_method = imputation_comparison.iloc[0]["method"]
print(f"Selected method from local masked-cell validation: {best_method}")

In [ ]:
# Fit the locally selected deployable imputer on all cleaned triage and later-ED vital measurements.
imputation_input = vitals_cleaned.copy()
for column in source_vitals:
    if imputation_input[column].notna().sum() == 0:
        imputation_input[column] = df[column]

if best_method == "autoencoder":
    raise ValueError("The autoencoder is comparison-only here; rerun validation with a deployable imputer selected.")

if best_method == "iterative":
    selected_imputer = IterativeImputer(
        random_state=0,
        max_iter=10,
        min_value=imputation_min_values,
        max_value=imputation_max_values,
    )
else:
    selected_imputer = imputers[best_method]

selected_values = pd.DataFrame(
    selected_imputer.fit_transform(imputation_input),
    columns=source_vitals,
    index=df.index,
)

# Keep raw measurements and add explicit method-suffixed columns for every vital.
imputed_triage_columns = []
for column in source_vitals:
    output_column = f"{column}_{best_method}"
    df[output_column] = selected_values[column]
    imputed_triage_columns.append(output_column)

triage_vitals = [
    "triage_temperature", "triage_heartrate", "triage_resprate",
    "triage_o2sat", "triage_sbp", "triage_dbp", "triage_pain", "triage_acuity",
]

after_imputation = pd.DataFrame({
    "raw_variable": source_vitals,
    "imputed_variable": imputed_triage_columns,
    "missing_before": [int(df[column].isna().sum()) for column in source_vitals],
    "missing_after": [int(df[f"{column}_{best_method}"].isna().sum()) for column in source_vitals],
    "raw_min": [df[column].min() for column in source_vitals],
    "imputed_min": [df[f"{column}_{best_method}"].min() for column in source_vitals],
    "raw_max": [df[column].max() for column in source_vitals],
    "imputed_max": [df[f"{column}_{best_method}"].max() for column in source_vitals],
})
display(after_imputation.round(2))

assert all(df[f"{column}_{best_method}"].notna().all() for column in source_vitals)
print(f"Raw vital columns remain available; _{best_method} columns are complete estimates for triage and later ED measurements.")

In [ ]:
#df

## 6. Quality checks: flag, inspect, document

The checks below do not silently repair the source CSV. They identify values for discussion. Negative duration is impossible as a stay length and will be excluded only from the LOS analysis; zero duration is retained and investigated as a possible real or administrative event.

In [ ]:
quality_checks = pd.DataFrame({
    "check": [
        "negative ED LOS", "zero ED LOS", "missing stay_id", "duplicate stay_id",
        "missing critical target", "missing hospitalization target", "age outside 0-120",
        "triage oxygen saturation outside 50-100",
    ],
    "count": [
        int(df["ed_los_hours"].lt(0).sum()), int(df["ed_los_hours"].eq(0).sum()),
        int(df["stay_id"].isna().sum()), int(df["stay_id"].duplicated().sum()),
        int(df["outcome_critical"].isna().sum()), int(df["outcome_hospitalization"].isna().sum()),
        int((df["age"].notna() & ~df["age"].between(0, 120)).sum()), int((df["triage_o2sat"].notna() & ~df["triage_o2sat"].between(50, 100)).sum()),
    ],
})
display(quality_checks)

In [ ]:
plot_los = df.loc[df["ed_los_hours"].between(0, 36), "ed_los_hours"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(plot_los, bins=72, color="#ea580c", ax=axes[0])
axes[0].set(title="ED LOS values from 0 to 36 hours", xlabel="ED LOS (hours)", ylabel="Stays")
sns.histplot(df["age"].dropna(), bins=40, color="#0d9488", ax=axes[1])
axes[1].set(title="Age distribution", xlabel="Age (years)", ylabel="Stays")
plt.tight_layout()
plt.show()

## 7. Outcome definitions and prevalence

The same row can support different questions, but the labels are not interchangeable:

- `ed_los_hours` is a continuous duration from ED arrival to ED departure.
- `outcome_critical` is the composite of in-hospital mortality and ICU transfer within 12 hours.
- `outcome_hospitalization` is true when `hadm_id` is present.

These definitions are retrospective labels. They do not mean that the outcome was known at triage.

In [ ]:
hospitalization_from_hadm = df["hadm_id"].notna()
critical_from_components = df["outcome_inhospital_mortality"] | df["outcome_icu_transfer_12h"]
assert (hospitalization_from_hadm == df["outcome_hospitalization"]).all()
assert (critical_from_components == df["outcome_critical"]).all()
target_summary = pd.DataFrame({
    "target": ["valid ED LOS", "critical outcome", "hospitalization"],
    "positive_or_valid_count": [int(df["ed_los_hours"].ge(0).sum()), int(df["outcome_critical"].sum()), int(df["outcome_hospitalization"].sum())],
    "percent_of_rows": [df["ed_los_hours"].ge(0).mean() * 100, df["outcome_critical"].mean() * 100, df["outcome_hospitalization"].mean() * 100],
})
display(target_summary.round(2))
component_summary = pd.DataFrame({
    "event": ["in-hospital mortality", "ICU transfer within 12 hours", "critical composite"],
    "count": [int(df["outcome_inhospital_mortality"].sum()), int(df["outcome_icu_transfer_12h"].sum()), int(df["outcome_critical"].sum())],
    "prevalence_pct": [df["outcome_inhospital_mortality"].mean() * 100, df["outcome_icu_transfer_12h"].mean() * 100, df["outcome_critical"].mean() * 100],
})
display(component_summary.round(2))

## 8. Descriptive comparisons are not predictions

It is reasonable to compare outcomes by triage acuity or arrival mode while exploring the data. It is not reasonable to call that comparison a causal effect, and it is not automatically a valid prediction model. The table below is a descriptive question: how do observed outcomes vary across acuity groups?

In [ ]:
acuity_summary = (
    df.groupby("triage_acuity", dropna=False)
    .agg(
        stays=("stay_id", "size"),
        median_los_hours=("ed_los_hours", "median"),
        critical_pct=("outcome_critical", "mean"),
        hospitalization_pct=("outcome_hospitalization", "mean"),
    )
    .reset_index()
)
acuity_summary[["critical_pct", "hospitalization_pct"]] *= 100
display(acuity_summary.round(2))

In [ ]:
df

In [ ]:
analysis_numeric = [
    "age", "triage_acuity_iterative", "triage_heartrate_iterative", "triage_sbp_iterative", "triage_o2sat_iterative",
    "n_ed_365d", "n_hosp_365d", "n_icu_365d", "ed_los_hours","triage_temperature_celsius_iterative"
]
correlation = df[analysis_numeric].corr(method="spearman")
plt.figure(figsize=(9, 7))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Spearman correlations for selected numeric fields")
plt.tight_layout()
plt.show()

## 9. Prediction boundary and leakage checklist

For the outcome models, the declared input boundary is triage-time information plus prior utilization, complaint flags, and comorbidity flags. The following are excluded: `stay_id`, `subject_id`, `hadm_id`, all timestamps, `disposition`, ED departure fields, `ed_los_hours`, `outcome_*`, ICU-transfer timing, revisit outcomes, and later ED measurements or medication counts.

Exclusion is not a claim that these fields are unimportant. It is a claim about when the information is available and what question the model is meant to answer.

In [ ]:
leakage_audit = pd.DataFrame({
    "field_or_family": [
        "stay_id / subject_id / hadm_id", "timestamps", "disposition", "ed_los_hours",
        "outcome_*", "time_to_icu_transfer_hours", "ed_*_last / medications",
    ],
    "why_excluded": [
        "identifiers or admission linkage; risk of memorization or target construction",
        "not part of a generic triage snapshot",
        "disposition is downstream of the ED decision",
        "the LOS label itself",
        "labels or downstream events",
        "event timing is not known at triage",
        "later measurements and treatment workflow",
    ],
})
display(leakage_audit)

## Student investigation before leaving M1

Choose one variable from three different families. For each variable, record its type, missing percentage, a useful summary or plot, the earliest time it can be known, and one limitation. Then answer: **Which of the three outcomes would this variable be allowed to predict, and why?**

Do not “clean” a value without preserving the raw table and writing down the rule. The handoff is a data-quality argument, not merely a new CSV.

In [ ]:
M1_tables = {
    "M1_variable_dictionary.csv": variable_dictionary,
    "M1_missingness.csv": missingness.reset_index(names="variable"),
    "M1_imputation_comparison.csv": imputation_comparison,
    "M1_vital_imputation_audit.csv": after_imputation,
    "M1_quality_checks.csv": quality_checks,
    "M1_target_summary.csv": target_summary,
    "M1_acuity_summary.csv": acuity_summary,
}
imputed_vitals_export = df[["stay_id"] + triage_vitals + imputed_triage_columns]
M1_tables["M1_triage_vitals_iter.csv"] = imputed_vitals_export
for filename, table in M1_tables.items():
    table.to_csv(OUTPUT_DIR / filename, index=False)
print(f"Wrote {len(M1_tables)} M1 handoff tables to {OUTPUT_DIR}")

In [ ]:
NEXT_MODULE_DATASET = OUTPUT_DIR / "M1_dataset_for_next_module.csv"
df.to_csv(NEXT_MODULE_DATASET, index=False)

print(f"Exported latest M1 dataset: {NEXT_MODULE_DATASET}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

## M1 takeaways

The table is an ED-stay dataset, not a patient table and not a sequence of isolated measurements. Basic statistics must respect skew and missingness. We compared source-informed imputation methods on artificially masked vital signs, selected iterative imputation on the local evidence, and preserved both the raw columns and the `_iter` audit trail. The three outcomes have different meanings and different leakage boundaries. The next analysis can therefore start from a declared question rather than a column guessed from its name.